# SuperTuxKart segmentation — frozen SAM 2 + a 2.1M decoder

Run the trained model on your own images, on a free Colab GPU.

The encoder (SAM 2 Hiera-Tiny, 27.2M params) stays frozen and is pulled from Hugging Face.
Only the decoder was trained — 2.07M parameters, **7% of the network**, an 8 MB checkpoint.

Validation mIoU **0.8897** over six classes; held-out test **0.7731**. The gap between those
two numbers is the point of the project — see the
[README](https://github.com/PabloAndresE/sam2-supertuxkart-segmentation).

**Runtime → Change runtime type → T4 GPU** makes this faster, but CPU works too.


## 1. Install and fetch the model


In [ ]:
!pip -q install 'transformers>=4.56' pyyaml
!git clone -q --depth 1 https://github.com/PabloAndresE/sam2-supertuxkart-segmentation.git repo
!wget -q -O repo/decoder_best.pt https://github.com/PabloAndresE/sam2-supertuxkart-segmentation/releases/download/v0.1.0/decoder_best.pt

import sys; sys.path.insert(0, '/content/repo/src')
print('decoder downloaded')


## 2. Load the model

The checkpoint holds only the decoder. The frozen encoder is downloaded separately,
which is why the artifact is 8 MB instead of 117 MB.


In [ ]:
import torch
from stkseg.registry import build_model
from stkseg.config import Config

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = Config.from_yaml('/content/repo/configs/default.yaml')

model = build_model(cfg.model).to(DEVICE)
model.load_decoder('/content/repo/decoder_best.pt', map_location=DEVICE)
model.eval()

print(f'device: {DEVICE}')
print(f'parameters: {model.param_counts}')


## 3. Segment the bundled example


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from stkseg.constants import CLASS_NAMES, COLORS
from stkseg.predict import segment

def show(image, alpha=0.55):
    mask, rgb = segment(model, image, cfg.data.img_size, DEVICE)
    coloured = COLORS[mask]
    # Background is black in the palette; tinting it would just darken the frame.
    blended = rgb.copy()
    region = mask != 0
    blended[region] = (rgb[region] * (1 - alpha) + coloured[region] * alpha).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, img, title in zip(axes, (rgb, coloured, blended), ('input', 'segmentation', 'overlay')):
        ax.imshow(img); ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()

    total = mask.size
    for c in np.unique(mask):
        print(f'  {CLASS_NAMES[c]:12s} {100 * (mask == c).sum() / total:6.2f}% of pixels')
    return mask

_ = show(Image.open('/content/repo/docs/assets/examples/abyss_frame_0100.png'))


## 4. Try your own image

Any SuperTuxKart frame works. Note the model was trained on frames rendered through
`pystk` at fixed graphics settings — screenshots from the retail game are a different
domain and have not been evaluated, so expect it to do worse on those.


In [ ]:
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    print(f'--- {name} ---')
    _ = show(Image.open(name))


## 5. Where the numbers come from

| Finding | Value |
|---|---|
| Selection score overstated generalisation by | 0.117 |
| Unseen scenery costs | 0.157 ± 0.090 |
| Which track is held out, vs initialisation noise | 16× |

Full protocol, ablations and the hypotheses that were measured and discarded:
[github.com/PabloAndresE/sam2-supertuxkart-segmentation](https://github.com/PabloAndresE/sam2-supertuxkart-segmentation)
